In [ ]:
from itertools import chain
import json
import os
from typing import Any, Hashable, Tuple

import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm import tqdm

from kibad_llm.config import PROJ_ROOT
from kibad_llm.dataset.json import read_and_preprocess
from kibad_llm.dataset.utils import merge_references_into_predictions
from kibad_llm.utils.dictionary import flatten_dict_simple

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)

In [ ]:
predictions_file = "data/results/2025-11-25_19-49-41/predictions.jsonl"
ground_truth_file = "data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"
FLATTEN_DICTS = True

In [ ]:
# this below is the exact same code used in the evaluation workflow


def load_predictions_with_references(
    pred_file: str, ref_file: str
) -> dict[Hashable, dict[str, dict]]:
    predictions = read_and_preprocess(
        file=pred_file,
        id_key="file_name",
        process_id=lambda x: os.path.splitext(x)[0],
        preprocess=lambda x: x.get("structured", None),
    )
    references = read_and_preprocess(
        file=ref_file,
        id_key="zotitem_ptr_id",
    )
    result = merge_references_into_predictions(predictions, references)
    return result


dataset = load_predictions_with_references(predictions_file, ground_truth_file)
f"loaded {len(dataset)} predictions"

In [ ]:
if FLATTEN_DICTS:

    # flatten_dict_simple does not handle nested entries, so we need to call it for prediction and reference individually.
    # Also, prediction may be None for some documents, so we use "{}" in these cases.
    dataset = {
        k: {
            "prediction": flatten_dict_simple(v["prediction"] or {}),
            "reference": flatten_dict_simple(v["reference"]),
        }
        for k, v in dataset.items()
    }

In [ ]:
predictions_df = pd.DataFrame.from_dict({k: v["prediction"] for k, v in dataset.items()}).T
reference_df = pd.DataFrame.from_dict({k: v["reference"] for k, v in dataset.items()}).T

print(f"available columns:\n{sorted(set(reference_df.columns) & set(predictions_df.columns))}")

In [ ]:
def get_column(col: str, replace_nan: Any | None = None) -> pd.DataFrame:
    result = pd.concat({"prediction": predictions_df[col], "reference": reference_df[col]}, axis=1)
    if replace_nan is not None:
        # we can not use .fillna(), since that does not work with lists
        mask = result.isna()
        result[mask] = result[mask].map(lambda _: replace_nan)

    return result


get_column("taxa.german_name", [])

In [ ]:
def _as_list_or_empty(x):
    """
    Wandelt x in eine Liste um:
    - Wenn x eine Liste/Tuple/Set ist → Liste zurück
    - Wenn x NaN/None → leere Liste
    - Wenn x ein String oder Skalar → [x]
    """
    import numpy as np
    import pandas as pd

    if x is None:
        return []
    # Series oder np.ndarray
    if isinstance(x, (pd.Series, np.ndarray)):
        x_ret = list(x)
        return [e for e in x_ret if e is not None]
    # Liste/Tuple/Set → Liste
    if isinstance(x, (list, tuple, set)):
        x_ret = list(x)
        return [e for e in x_ret if e is not None and e != ""]
    # alles andere (z.B. String, Zahl) → einzelnelement-Liste
    if x == "":
        return []
    return [x]


def align_row(pred_list, true_list, missing_label="_NA_"):
    """
    Für eine Zeile: pred_list und true_list sind (möglicherweise) Listen von Strings.
    Gibt (union_sorted, aligned_pred, aligned_true) zurück:
      - union_sorted: alphabetisch sortierte Union der Labels
      - aligned_pred: Liste gleicher Länge; Label wenn in pred_list vorhanden, sonst None
      - aligned_true: analog für true_list
    """
    pred = [str(v) for v in _as_list_or_empty(pred_list)]
    true = [str(v) for v in _as_list_or_empty(true_list)]

    union = sorted(set(pred) | set(true))
    aligned_pred = [label if label in pred else missing_label for label in union]
    aligned_true = [label if label in true else missing_label for label in union]

    return union, aligned_pred, aligned_true


# -------------------------
# Beispiel-Anwendung: df_predictions und df_ground_truth haben dieselbe Reihenfolge / Index
# -------------------------

# Beispiel-DataFrames (nur zur Demonstration)
# df_predictions = pd.DataFrame({'taxa.german_name': [["Amsel","Meise"], ["Kohlmeise"], []]})
# df_ground_truth = pd.DataFrame({'taxa.german_name': [["Amsel"], ["Meise","Kohlmeise"], ["Buntspecht"]]})


# Sicherstellen, dass die Indizes zusammenpassen; ansonsten vorher joinen/merge
if not predictions_df.index.equals(reference_df.index):
    # wenn nicht, joinieren wir per default inner-join auf Index (alternative: reset_index+merge, oder auf ID-Spalte joinen)
    df = (
        predictions_df[["taxa.german_name"]]
        .rename(columns={"taxa.german_name": "y_pred"})
        .join(
            reference_df[["taxa.german_name"]].rename(columns={"taxa.german_name": "y_true"}),
            how="inner",
        )
    )
else:
    df = pd.DataFrame(
        {"y_pred": predictions_df["taxa.german_name"], "y_true": reference_df["taxa.german_name"]}
    )

# Zeilenweise anwenden und neue Spalten anlegen
res = df.apply(lambda row: align_row(row["y_pred"], row["y_true"]), axis=1)

# res ist eine Series mit Tupeln (union, aligned_pred, aligned_true) — wir entpacken:
df[["union_labels", "aligned_pred", "aligned_true"]] = pd.DataFrame(res.tolist(), index=res.index)

# Ergebnis: df enthält nun pro Zeile die union und die beiden aligned-Listen
# Beispiel-Ausgabe:
# df[['union_labels','aligned_pred','aligned_true']].head()


# Flatten der Listen-Spalten
union_labels_flat = sorted(list(set(list(chain.from_iterable(df["union_labels"])))))
# print(union_labels_flat)
y_pred_flat = list(chain.from_iterable(df["aligned_pred"]))
y_true_flat = list(chain.from_iterable(df["aligned_true"]))

# F1-Score berechnen (für Multiclass)
score = precision_recall_fscore_support(
    y_true_flat, y_pred_flat, average="micro", labels=union_labels_flat
)  # oder 'micro', 'weighted'
print("F1-Score:", score)

cm = confusion_matrix(y_true_flat, y_pred_flat, labels=union_labels_flat + ["_NA_"])
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=union_labels_flat + ["_NA_"])
disp.plot()

plt.show()

In [ ]:
# biodiversity_variable -> gold label sind oft Englisch, z.B. 'Abundance of mice', 'Abundance of voles'
# transformation_potential -> nie predicted
# in location sind gold werte wie '(('country', 'Australia'), ('federal_state', 'Baden-Württemberg'), ('name', 'Western Australian coast'))' ???
# in location sind gold werte wie (('country', 'Belgium'),) |   (('country', 'Cambodia'), ('name', 'Cambodia'))
# species mismatches wegen kleiner unterschiede, z.B predicted = ('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium L.'), ('species_group', 'Gefäßpflanzen')
# gold = (('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium'), ('species_group', 'Gefäßpflanzen')) -> einziger Unterschied ist ' L.' im scientific name
# ansonsten sehen predictions für taxa eigentlich gut aus (immer deutscher oder lateinischer Begriff)